## Training LLMs

### Pre-Training LLMs

In [ ]:
GPT_CONFIG_124M={
  "vocab_size": 50257,     # Vocabulary size
  "context_length": 256,  # Context length
  "emb_dim": 768,          # Embedding dimension
  "n_heads": 12,           # Number of attention heads
  "n_layers": 12,          # Number of transformers
  "drop_rate": 0.1,        # Dropout rate
  "qkv_bias": False        # Query-Key-Value bias
}

### Complete GPT Architecture

In [ ]:
import torch
import torch.nn as nn

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb=nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb=nn.Dropout(cfg["drop_rate"])

    self.trf_blocks=nn.Sequential(
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    self.final_norm=LayerNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self, in_idx):
    batch_size, seq_len=in_idx.shape
    tok_embeds=self.tok_emb(in_idx)
    pos_embeds=self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x=tok_embeds+pos_embeds
    x=self.drop_emb(x)
    x=self.trf_blocks(x)
    x=self.final_norm(x)
    logits=self.out_head(x)
    return logits
  
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      dropout=cfg["drop_rate"],
      num_heads=cfg["n_heads"],
      qkv_bias=cfg["qkv_bias"]
    )
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # Shortcut connection for attention block
    shortcut=x
    # Every row in the input has 0 mean and 1 variance
    x=self.norm1(x)
    # We get the context vector of [batch_size, num_tokens, emb_dim]
    x=self.att(x)
    # Dropout layer to improve efficiency
    x=self.drop_shortcut(x)
    # Creating shortcut connection
    x=x+shortcut

    # Shortcut for feed forward block
    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x=x+shortcut

    return x
  
import torch.nn as nn

class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out%num_heads==0), \
    "d_out must be divisible by num_heads"

    self.d_out=d_out
    self.num_heads=num_heads
    self.head_dim=d_out//num_heads

    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    # Linear layer to combine head outputs
    self.out_proj=nn.Linear(d_out, d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in=x.shape

    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    # We implicitly split the matrix by adding a `num_heads` dimension
    # Unroll last dimension: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries=queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values=values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys=keys.transpose(1, 2)
    queries=queries.transpose(1, 2)
    values=values.transpose(1, 2)

    # Computing attention scores
    attn_scores=torch.matmul(queries, keys.transpose(2, 3))

    # Original mask truncated to the number of tokens and converted to bool
    masked_bool=self.mask.bool()[:num_tokens, :num_tokens]

    # Using the mask to fill the attention scores
    masked_attn_scores=attn_scores.masked_fill_(masked_bool, -torch.inf)

    # Calculating the attention weights
    attn_weights=torch.softmax(masked_attn_scores/keys.shape[-1]**0.5, dim=-1)

    # Feeding attention weights to the dropout layer
    attn_weights=self.dropout(attn_weights)

    # Calculating context vectors
    context_vecs=(attn_weights@values).transpose(1, 2) # To get the original dimensions

    # Combining heads where self.d_out=num_heads*head_dim
    # contiguous - to make sure the reshaped matrices are in same blocks of memory
    context_vecs=context_vecs.contiguous().view(b, num_tokens, self.d_out)
    context_vecs=self.out_proj(context_vecs)

    return context_vecs
  
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean=x.mean(dim=-1, keepdim=True)
    var=x.var(dim=-1, keepdim=True, unbiased=False)
    # To prevent from zero division
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift
  
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  
  def forward(self, x):
    gelu=0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi, device=x.device))*(x+(0.044715*torch.pow(x, 3)))))
    return gelu
  
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers=nn.Sequential(
      # Expansion
      nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
      # Activation
      GELU(),
      # Contraction
      nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
  # idx is (batch_size, num_tokens) array of indices in current context
  for _ in range(max_new_tokens):
    # Crop current context if it exceeds the supported context size
    '''For example, 
    Case 1: If LLM supports only 5 tokens and context size is
    10, then only the last 5 tokens are used as context.
    Case 2: If LLM supports 8 tokens and context size is 5, then
    only the last 5 tokens are used as context.'''
    idx_cond=idx[:, -context_size:]

    # Get output tensors - (batch_size, num_tokens, vocab_size)
    with torch.no_grad():
      logits=model(idx_cond)

    # Extract last vector
    logits=logits[:, -1, :]

    # Apply softmax to get probabilities - (batch_size, vocab_size)
    probs=torch.softmax(logits, dim=-1)

    # Get the idx of the vocab entry with the highest probability value
    idx_next=torch.argmax(probs, dim=-1, keepdim=True)
    # (batch_size, 1)

    # Append sampled index to the running sequence
    idx=torch.cat((idx, idx_next), dim=1)
    # (batch_size, num_tokens+1)

  return idx

def text_to_token_ids(text, tokenizer):
  encoded_text=tokenizer.encode(text, allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded_text).unsqueeze(0)
  return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
  flat=token_ids.squeeze(0)
  decoded_text=tokenizer.decode(flat.tolist())
  return decoded_text

In [ ]:
torch.manual_seed(123)

model=GPTModel(GPT_CONFIG_124M)
model.eval()

In [ ]:
import tiktoken

start_context="Every effort moves you"
tokenizer=tiktoken.get_encoding('gpt2')

token_ids=generate_text_simple(
  model=model,
  idx=text_to_token_ids(start_context, tokenizer),
  max_new_tokens=10,
  context_size=GPT_CONFIG_124M["context_length"]
)

output_text=token_ids_to_text(token_ids, tokenizer)
print(f'Output Text: {output_text}')

### Measuring LLM Loss Function

In [ ]:
inputs=torch.tensor([[16833, 3626, 6100],  # [every, effort, moves]
                     [40, 1107, 588]])     # [I, really, like]

targets=torch.tensor([[3626, 6100, 345],   # [effort, moves, you]
                      [1107, 588, 11311]]) # [really, like, chocolate]

In [ ]:
with torch.no_grad():
  logits=model(inputs)

probs=torch.softmax(logits, dim=-1)
print(probs.shape)

In [ ]:
token_ids=torch.argmax(probs, dim=-1, keepdim=True)
print(f'Token IDs:\n{token_ids}')

In [ ]:
print(f'Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}')
# Converts multi-dimensional tensor to 1D tensor
print(f'Predicted batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}')

### Cross Entropy Loss

- Measures the difference between 2 probability distributions.

- First calculate the `logits`.

- Then get the `probabilities` by applying softmax on logits.

- `Negative Log Likelihood`
  - Now get the `target probabilities` using the targets IDs.

  - Apply `log` on all the target probabilities.

  - Get the `mean` of the logaithmic probabilities.

  - Apply `negative` to the mean.

In [ ]:
torch.set_printoptions(sci_mode=False)

text_idx=0
target_probs_1=probs[text_idx, [0, 1, 2], targets[text_idx]]
print(f'Text 1 probabilities: {target_probs_1}')

text_idx=1
target_probs_2=probs[text_idx, [0, 1, 2], targets[text_idx]]
print(f'Text 2 probabilities: {target_probs_2}')

In [ ]:
# Compute the logarithms of all token probabilities

log_probs=torch.log(torch.cat((target_probs_1, target_probs_2)))
print(log_probs)

In [ ]:
# Calculate the average probability for each token

avg_log_prob=torch.mean(log_probs)
print(avg_log_prob)

In [ ]:
neg_avg_log_prob=avg_log_prob*-1
print(neg_avg_log_prob)

In [ ]:
# Using the built-in function

print(logits.shape)
print(targets.shape)

logits_flat=logits.flatten(0, 1)
targets_flat=targets.flatten()

print(logits_flat.shape)
print(targets_flat.shape)

loss=nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

### Perplexity

- Measures how well the probability distribution predicted by the model matches the actual distribution of words in the dataset.

- More interpretable way of understanding model uncertainty in predicting next token.

- Lower perplexity score - better predictions.

In [ ]:
perplexity=torch.exp(loss)
print(perplexity)

Model is roughly as uncertain as if it had to choose the next token randomly from about 48725 tokens in the vocabulary.

### Training and Validation Losses

In [ ]:
with open("verdict.txt", "r", encoding='utf-8-sig') as f:
  text_data=f.read()

In [ ]:
tot_characters=len(text_data)
tot_tokens=len(tokenizer.encode(text_data))

print(f'Number of characters: {tot_characters}')
print(f'Number of tokens: {tot_tokens}')

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids=[]
    self.target_ids=[]

    # Tokenize the entire text
    token_ids=tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

    # Use a sliding window to chunk the book into overlapping sequences of max_length
    for i in range(0, len(token_ids)-max_length, stride):
      input_chunk=token_ids[i:i+max_length]
      target_chunk=token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)
  
  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]
  
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
  # Initialize the tokenizer
  tokenizer=tiktoken.get_encoding('gpt2')

  # Create dataset
  dataset=GPTDatasetV1(txt, tokenizer, max_length, stride)

  # Create dataloader
  dataloader=DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=shuffle,
    drop_last=drop_last,
    num_workers=num_workers
  )

  return dataloader

In [ ]:
# Training and Validation data

train_ratio=0.9
split_idx=int(train_ratio*len(text_data))
train_data=text_data[:split_idx]
val_data=text_data[split_idx:]

In [ ]:
torch.manual_seed(123)

# In GPT model, context size = stride
train_loader=create_dataloader_v1(
  train_data,
  batch_size=2,
  max_length=GPT_CONFIG_124M["context_length"],
  stride=GPT_CONFIG_124M["context_length"],
  shuffle=True,
  drop_last=True,
  num_workers=0
)

val_loader=create_dataloader_v1(
  val_data,
  batch_size=2,
  max_length=GPT_CONFIG_124M["context_length"],
  stride=GPT_CONFIG_124M["context_length"],
  shuffle=False,
  drop_last=False,
  num_workers=0
)

In [ ]:
# Sanity Check

if tot_tokens*train_ratio<GPT_CONFIG_124M["context_length"]:
  print("Not enough tokens for the training loader."
        "Try to lower the context length or" \
        "increase the training ratio.")
  
if tot_tokens*(1-train_ratio)<GPT_CONFIG_124M["context_length"]:
  print("Not enough tokens for the validation loader."
        "Try to lower the context length or" \
        "decrease the training ratio.")

In [ ]:
print('Training loader:')
for x, y in train_loader:
  print(x.shape, y.shape)

print("Validation loader:")
for x, y in val_loader:
  print(x.shape, y.shape)
  
print(len(train_loader))

In [ ]:
train_tokens=0
for input_batch, target_batch in train_loader:
  train_tokens+=input_batch.numel()

val_tokens=0
for input_batch, target_batch in val_loader:
  val_tokens+=input_batch.numel()

print(f'Training tokens: {train_tokens}')
print(f'Validation tokens: {val_tokens}')
print(f'Total tokens: {train_tokens+val_tokens}')

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
  input_batch, target_batch=input_batch.to(device), target_batch.to(device)
  logits=model(input_batch)
  loss=nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
  return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
  tot_loss=0

  if len(data_loader)==0:
    return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches, len(data_loader))

  for i, (input_batch, target_batch) in enumerate(data_loader):
    if i<num_batches:
      loss=calc_loss_batch(input_batch, target_batch, model, device)
      tot_loss+=loss.item()
    else:
      break

  return tot_loss/num_batches

In [ ]:
if torch.cuda.is_available():
  device=torch.device("cuda")
elif torch.backends.mps.is_available():
  device=torch.device("mps")
else:
  device=torch.device("cpu")

print(f"Device used: {device}")

model.to(device)

This code selects the **best available hardware accelerator** for PyTorch computations

#### Summary:

* **`cuda`**: For systems with NVIDIA GPUs.

* **`mps`**: For Apple devices with M1/M2 chips.

* **`cpu`**: Default when no GPU acceleration is available.

This allows your code to run efficiently on different hardware without manual changes.

In [ ]:
torch.manual_seed(123)

with torch.no_grad():
  train_loss=calc_loss_loader(train_loader, model, device)
  val_loss=calc_loss_loader(val_loader, model, device)

print(f'Training Loss: {train_loss}')
print(f'Validation Loss: {val_loss}')

### LLM Training Loop

In [ ]:
def train_model_sample(model, train_loader, val_loader, optimizer,
                       device, num_epochs, eval_freq, eval_iter,
                       start_context, tokenizer):
  # Initialize lists to track losses and tokens seen
  train_losses, val_losses, track_tokens_seen=[], [], []
  tokens_seen, global_step=0, -1

  # Main training loop
  for epoch in range(num_epochs):
    # Set model to training mode
    model.train()

    for input_batch, target_batch in train_loader:
      # Reset loss gradients from previous batch iteration
      optimizer.zero_grad()
      loss=calc_loss_batch(input_batch, target_batch, model, device)
      # Calculate loss gradients
      loss.backward()
      # Update model weights using loss gradients
      optimizer.step()
      # Return the total number of elements in the input batch
      tokens_seen+=input_batch.numel()
      global_step+=1

      # Optional evaluation step
      if global_step%eval_freq==0:
        train_loss, val_loss=evaluate_model(
          model, train_loader, val_loader, device, eval_iter
        )
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        track_tokens_seen.append(tokens_seen)
        print(f'Epoch {epoch+1} (Step {global_step:06d}): '
              f'Train Loss: {train_loss:.3f}, Val Loss: {val_loss:.3f}')
        
    generate_and_print_sample(
      model, tokenizer, device, start_context
    )

  return train_losses, val_losses, track_tokens_seen

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
  model.eval()
  with torch.no_grad():
    train_loss=calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
    val_loss=calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
  model.train()
  return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
  model.eval()
  context_size=model.pos_emb.weight.shape[0]
  encoded=text_to_token_ids(start_context, tokenizer).to(device)
  with torch.no_grad():
    token_ids=generate_text_simple(
      model, encoded, max_new_tokens=50, context_size=context_size
    )
  decoded=token_ids_to_text(token_ids, tokenizer)
  print(decoded.replace('\n', " "))
  model.train()

In [ ]:
import time

start_time=time.time()
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
model.to(device)
# weight_decay prevents overfitting
optimizer=torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs=10
train_losses, val_losses, tokens_seen=train_model_sample(
  model=model,
  train_loader=train_loader,
  val_loader=val_loader,
  optimizer=optimizer,
  device=device,
  num_epochs=num_epochs,
  eval_freq=5,
  eval_iter=10,
  start_context="Every effort moves you",
  tokenizer=tokenizer
)

end_time=time.time()
exec_time_min=(end_time-start_time)/60
print(f'Training completed in {exec_time_min:.2f} minutes.')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
  fig, ax1=plt.subplots(figsize=(5, 3))

  # Plot training and validation loss against epochs
  ax1.plot(epochs_seen, train_losses, label="Training loss")
  ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
  ax1.set_xlabel("Epochs")
  ax1.set_ylabel("Loss")
  ax1.legend(loc="upper right")
  # Show only integer labels on x axis
  ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

  # Plot training and validation loss against tokens seen
  ax2=ax1.twiny()
  ax2.plot(tokens_seen, train_losses, alpha=0)
  ax2.set_xlabel("Tokens")

  fig.tight_layout()
  plt.show()

epochs_tensor=torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

### Decoding Strategies to control Randomness - Text Generation Strategies

#### Temperature Scaling

- Until now, the generated token is selected corresponding to the longest probability score among all tokens in the vocabulary with `torch.argmax` which is known as `greedy decoding`.

- This leads to lot of randomness and diversity in generated text.

- `Temperature Scaling`: Replace argmax with `multinomial probability distribution` which samples next token according to probability score (LLM generates for each vocabulary entry at each token generation step).

In [ ]:
model.to("cpu")
model.eval()

In [ ]:
vocab={
  "closer": 0,
  "every": 1,
  "effort": 2,
  "forward": 3,
  "inches": 4,
  "moves": 5,
  "pizza": 6,
  "toward": 7,
  "you": 8
}

inv_vocab={v:k for k, v in vocab.items()}

In [ ]:
next_token_logits=torch.tensor(
  [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

In [ ]:
probs=torch.softmax(next_token_logits, dim=0)
print(probs)

next_token_id=torch.argmax(probs).item()
print(next_token_id)

print(inv_vocab[next_token_id])

In [ ]:
torch.manual_seed(123)

next_token_id=torch.multinomial(probs, num_samples=1).item()
print(inv_vocab[next_token_id])

- The `multinomaial` function samples the next token proportional to its probability score i.e; maximum probability.

In [ ]:
def print_sampled_tokens(probs):
  torch.manual_seed(123)
  sample=[torch.multinomial(probs, num_samples=1).item() for i in range(1000)]
  # Count the frequency of each value in an array of non-negative ints
  sampled_ids=torch.bincount(torch.tensor(sample))
  for i, freq in enumerate(sampled_ids):
    print(f'{freq} x {inv_vocab[i]}')

print_sampled_tokens(probs)

- Temperature scaling is fancy term for dividing the logits by a number greater than 0.

- `Scaled logits=logits/temperature`

- If `temperature` is small, then sharper distribution else flattened distribution.

In [ ]:
def softmax_with_temp(logits, temperature):
  scaled_logits=logits/temperature
  return torch.softmax(scaled_logits, dim=0)

temp_values=[1, 0.1, 5]

scaled_probs=[softmax_with_temp(next_token_logits, T) for T in temp_values]
print(scaled_probs)

In [ ]:
x=torch.arange(len(vocab))
bar_width=0.15

fig, ax=plt.subplots(figsize=(5, 3))
for i, T in enumerate(temp_values):
  rects=ax.bar(x+i*bar_width, scaled_probs[i], bar_width, label=f'Temperature={T}')

ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()

plt.tight_layout()
plt.show()

- Applying `very small temperatures` will result in sharper distributions such that the behaviour of the multinomial function selects the most likely token approaching the behaviour of the argmax function.

- The issue with temperature scaling is that higher temperature values result in more uniformly distributed next-token probabilities, which result in more diverse outputs as it reduces the likelihood of the model repeatedly selecting the most probable token.

- This method allows for exploring less likely but potentially more interesting and creative paths in the generation process.

- This approach sometimes leads to grammatically incorrect or completely non-sensical outputs.

#### Top-k Sampling

- Restrict the sampled tokens to the top-k most likely tokens and exclude all other tokens.

- Here first consider only top k sampled tokens in the logits and apply `-inf` mask to all other tokens and then apply `softmax`.

- By assigning zero probabilities to the non top-k positions, we ensure that the next token is always sampled from a top-k position.

In [ ]:
top_k=3
top_k_logits, top_k_pos=torch.topk(next_token_logits, top_k)
print(f'Top logits: {top_k_logits}')
print(f'Top positions: {top_k_pos}')

In [ ]:
# Return a tensor of elements selected from either input or other, depending on condition
new_logits=torch.where(
  condition=next_token_logits<top_k_logits[-1],
  input=torch.tensor(float("-inf")),
  other=next_token_logits
)

print(new_logits)

In [ ]:
new_logits_probs=torch.softmax(new_logits, dim=0)
print(new_logits_probs)

#### Merging Temperature scaling and Top-k sampling

- Take `logits`.

- Define `k` and consider `top-k` elements in the logits using `torch.topk`.

- Apply `-inf mask` to the logits using `torch.where`.

- Apply temperature scaling by dividing the logits with `temp`.

- Apply the `softmax` on the masked top-k logits.

- Finally sample the probabilities using `torch.multinomial` to get the probability scores of the top-k logits.

- This ensures that there is some amount of creativity in the process and we ensure random tokens are not given the oppurtunity to become next token.

- This will make sure `overfitting` is reduced.

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temp=0.0, top_k=None, eos_id=None):
  for _ in range(max_new_tokens):
    idx_cond=idx[:, -context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:, -1, :]

    # Filter logits with top-k sampling
    if top_k is not None:
      top_k_logits, _=torch.topk(logits, top_k)
      min_val=top_k_logits[:, -1]
      logits=torch.where(
        condition=logits<min_val,
        input=torch.tensor(float("-inf")).to(device),
        other=logits
      )

    # Apply temperature scaling
    if temp>0.0:
      logits=logits/temp
      probs=torch.softmax(logits, dim=-1)
      idx_next=torch.multinomial(probs, num_samples=1)
    else:
      idx_next=torch.argmax(probs, dim=-1, keepdim=True)

    # Stop generating early if end-of-sequence token is encountered
    if idx_next==eos_id:
      break
      
    idx=torch.cat((idx, idx_next), dim=1)

  return idx

In [ ]:
torch.manual_seed(123)

token_ids=generate(
  model=model,
  idx=text_to_token_ids("Every effort moves you", tokenizer),
  max_new_tokens=20,
  context_size=GPT_CONFIG_124M["context_length"],
  temp=1.4,
  top_k=25
)

print(f'Output text: {token_ids_to_text(token_ids, tokenizer)}')

### Saving and Loading PyTorch Model Weights

- `torch.save(model.state_dict(), "model.pth")`

  - `state_dict`: dictionary mapping each layer to its parameter

  - `model.pth`: file where state_dict is saved

- `model.load_state_dict(torch.load("model.pth"))`

- `Optimizer` also contains the history of gradients and squared gradients.

- `optimizer_state_dict` will save the hyperparameters and historical data such as past gradients.

- If we don't save the optimizer data, then when we logout, optimizer resets and the model may learn suboptimally or even fail to converge properly, which may lead to loosing the ability to generate coherant text.

In [ ]:
model=GPTModel(GPT_CONFIG_124M)
# .pth extension is convention for PyTorch files
torch.save(model.state_dict(), "model.pth")

In [ ]:
model=GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth"))
model.eval()

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

torch.save({
  "model_state_dict": model.state_dict(),
  "optimizer_state_dict": optimizer.state_dict()
}, "model_and_optimizer.pth")

In [ ]:
checkpoint=torch.load("model_and_optimizer.pth")

model=GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer=torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train()

### Loading Pre-Trained OpenAI Weights

- OpenAI originally saved the GPT-2 weights via Tensorflow, which we need to load the weights in Python.

- `tqdm` is a progress bar tool to track the download progress.

In [ ]:
import tensorflow as tf
import tqdm

print(tf.__version__)
print(tqdm.__version__)

### gpt_download Overview

---

#### ✅ 1. `download_and_load_gpt2(model_size, models_dir)`

#### ✅ Purpose:

* Downloads all necessary files for the specified GPT-2 model size from OpenAI's storage.
* Loads the model’s settings from `hparams.json` and parses TensorFlow checkpoint files into Python structures.

#### ✅ Parameters:

* `model_size` (str): One of `"124M"`, `"355M"`, `"774M"`, `"1558M"`.
* `models_dir` (str): Directory path to store model files (e.g., `./models`).

#### ✅ Steps:

1. **Validation**:

   * Ensures `model_size` is one of the four allowed sizes.
2. **Setup**:

   * Constructs a `model_dir` using the `models_dir` and `model_size`.
   * Lists all required filenames (e.g., `checkpoint`, `model.ckpt.*`, etc.).
3. **Download**:

   * Ensures the directory exists.
   * Iterates over all filenames and calls `download_file(...)` to fetch and store them locally.
4. **Load TensorFlow Checkpoint**:

   * Uses `tf.train.latest_checkpoint(...)` to get the path to the latest `.ckpt` file.
   * Reads hyperparameters from `hparams.json`.
   * Calls `load_gpt2_params_from_tf_ckpt(...)` to convert checkpoint weights into a nested dictionary.

#### ✅ Returns:

* `settings`: Dictionary from `hparams.json` (model hyperparameters).
* `params`: Nested dictionary structure of model weights extracted from TensorFlow checkpoint.

---

#### ✅ 2. `download_file(url, destination)`

#### ✅ Purpose:

Downloads a file from a URL and saves it to a specified path with progress feedback using `tqdm`.

#### ✅ Parameters:

* `url` (str): The file's URL.
* `destination` (str): Path to store the downloaded file locally.

#### ✅ Steps:

1. Sends a `GET` request to download the file with `verify=False` (SSL check is disabled — not recommended for production).
2. Checks if the file already exists and matches the expected size (`content-length` from headers). If so, skips download.
3. Uses a block size of `1024` bytes (1 KB) and streams the content to the file while showing a progress bar using `tqdm`.

#### ✅ Error Handling:

* Catches any network errors (e.g., no internet, invalid URL).
* Prints a helpful message if the download fails.

---

#### ✅ 3. `load_gpt2_params_from_tf_ckpt(ckpt_path, settings)`

#### ✅ Purpose:

Parses GPT-2's TensorFlow checkpoint `.ckpt` files into a clean and nested Python dictionary for easy access.

#### ✅ Parameters:

* `ckpt_path`: Path to the checkpoint directory.
* `settings`: The hyperparameters dictionary (from `hparams.json`) used to determine number of layers, etc.

#### ✅ Steps:

1. **Initialize `params` dictionary**:

   ```python
   params = {"blocks": [{} for _ in range(settings["n_layer"])]}
   ```

   This prepares one dictionary for each transformer layer block (`h0`, `h1`, ..., `hn`).

2. **Iterate over all variables in the checkpoint** using `tf.train.list_variables(...)`.

3. For each variable:

   * **Load it** using `tf.train.load_variable(...)` and `np.squeeze()` to remove unnecessary dimensions.
   * **Parse variable name** like: `model/h0/attn/c_attn/w` → \['h0', 'attn', 'c\_attn', 'w'].
   * **Determine where to store it**:

     * If it's a layer variable (starts with `h0`, `h1`, etc.), store it in `params["blocks"][layer_number]`.
     * Otherwise, keep it in the root `params` dictionary.
   * **Recursively build nested dictionary structure** using `setdefault()` to store deeper keys.

#### ✅ Returns:

* `params`: Dictionary with all weights organized by transformer blocks and components (attention, MLP, etc.).

---

### 🔁 Example Workflow:

```python
settings, params = download_and_load_gpt2("124M", "./models")
```

### ✅ Summary

| Function                        | Purpose                                                              |
| ------------------------------- | -------------------------------------------------------------------- |
| `download_and_load_gpt2`        | Coordinates download and loading of GPT-2 model files                |
| `download_file`                 | Downloads individual files with progress bar and integrity checks    |
| `load_gpt2_params_from_tf_ckpt` | Converts raw TensorFlow checkpoint into structured Python dictionary |

In [ ]:
from gpt_download import download_and_load_gpt2

settings, params=download_and_load_gpt2(
  model_size="124M",
  models_dir="gpt2"
)

#### Parameter Dictionary Keys

- `wte`: Token Embeddings - 50257 x 768

- `wpe`: Positional Embeddings - 1024 x 768

- `blocks`:
  
  * `Attention layers in transformer blocks`:
    
    * transformer/h0/attn/c_attn/w
    * transformer/h0/attn/c_attn/b
    * Similarly for h1, h2....h11

  * `Feed Forward neural network weights in transformer blocks`:

    * transformer/h0/mlp/c_fc/w
    * transformer/h0/mlp/c_fc/b
    * Similarly for h1, h2....h11

    * transformer/h0/mlp/c_proj/w
    * transformer/h0/mlp/c_proj/b
    * Similarly for h1, h2....h11

  * `Output projection layer in transformer blocks`:
    
    * transformer/h0/attn/c_proj/w
    * Similarly for h1, h2....h11

  * `Layer normalization in transformer blocks`:

    * `Layer 1`:

      * transformer/h0/ln_1/g -> layer norm scale
      * transformer/h0/ln_1/b -> layer norm shift
      * Similarly for h1, h2....h11

    * `Layer 2`:

      * transformer/h0/ln_2/g -> layer norm scale
      * transformer/h0/ln_2/b -> layer norm shift
      * Similarly for h1, h2....h11

- `g`: Final norm scale

- `b`: Final norm shift

#### Key Terms:

1. `c_attn`: Fused matrix of query, key, value matrices in attn of particular transformer.

2. `mlp`: Multi-layer perceptron.

3. `fc`: Fully connected layer - Expansion in feed forward.

4. `proj`: Projection layer - Contraction in feed forward.

In [ ]:
print(f'Settings: {settings}')
print(f'Parameter Dictionary Keys: {params.keys()}')

In [ ]:
print(params["wte"].shape)

In [ ]:
# Model configurations

model_configs={
  "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
  "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
  "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
  "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25}
}

model_name="gpt2-small (124M)"
NEW_CONFIG=GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])

In [ ]:
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})
gpt=GPTModel(NEW_CONFIG)
gpt.eval()

### Updating Random weights to OpenAI weights

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
  gpt.pos_emb.weight=assign(gpt.pos_emb.weight, params["wpe"])
  gpt.tok_emb.weight=assign(gpt.tok_emb.weight, params["wte"])

  for b in range(len(params["blocks"])):
    q_w, k_w, v_w=np.split(
      params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
    gpt.trf_blocks[b].att.w_query.weight=assign(
      gpt.trf_blocks[b].att.w_query.weight, q_w.T
    )
    gpt.trf_blocks[b].att.w_key.weight=assign(
      gpt.trf_blocks[b].att.w_key.weight, k_w.T
    )
    gpt.trf_blocks[b].att.w_value.weight=assign(
      gpt.trf_blocks[b].att.w_value.weight, v_w.T
    )

    q_b, k_b, v_b=np.split(
      params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
    gpt.trf_blocks[b].att.w_query.bias=assign(
      gpt.trf_blocks[b].att.w_query.bias, q_b
    )
    gpt.trf_blocks[b].att.w_key.bias=assign(
      gpt.trf_blocks[b].att.w_key.bias, k_b
    )
    gpt.trf_blocks[b].att.w_value.bias=assign(
      gpt.trf_blocks[b].att.w_value.bias, v_b
    )

    gpt.trf_blocks[b].att.out_proj.weight=assign(
      gpt.trf_blocks[b].att.out_proj.weight,
      params["blocks"][b]["attn"]["c_proj"]["w"].T
    )
    gpt.trf_blocks[b].att.out_proj.bias=assign(
      gpt.trf_blocks[b].att.out_proj.bias,
      params["blocks"][b]["attn"]["c_proj"]["b"]
    )

    gpt.trf_blocks[b].ff.layers[0].weight=assign(
      gpt.trf_blocks[b].ff.layers[0].weight,
      params["blocks"][b]["mlp"]["c_fc"]["w"].T
    )
    gpt.trf_blocks[b].ff.layers[0].bias=assign(
      gpt.trf_blocks[b].ff.layers[0].bias,
      params["blocks"][b]["mlp"]["c_fc"]["b"]
    )
    gpt.trf_blocks[b].ff.layers[2].weight=assign(
      gpt.trf_blocks[b].ff.layers[2].weight,
      params["blocks"][b]["mlp"]["c_proj"]["w"].T
    )
    gpt.trf_blocks[b].ff.layers[2].bias=assign(
      gpt.trf_blocks[b].ff.layers[2].bias,
      params["blocks"][b]["mlp"]["c_proj"]["b"]
    )

    gpt.trf_blocks[b].norm1.scale=assign(
      gpt.trf_blocks[b].norm1.scale,
      params["blocks"][b]["ln_1"]["g"]
    )
    gpt.trf_blocks[b].norm1.shift=assign(
      gpt.trf_blocks[b].norm1.shift,
      params["blocks"][b]["ln_1"]["b"]
    )
    gpt.trf_blocks[b].norm2.scale=assign(
      gpt.trf_blocks[b].norm2.scale,
      params["blocks"][b]["ln_2"]["g"]
    )
    gpt.trf_blocks[b].norm1.shift=assign(
      gpt.trf_blocks[b].norm1.shift,
      params["blocks"][b]["ln_2"]["b"]
    )

  gpt.final_norm.scale=assign(gpt.final_norm.scale, params["g"])
  gpt.final_norm.shift=assign(gpt.final_norm.shift, params["b"])
  gpt.out_head.weight=assign(gpt.out_head.weight, params["wte"])

def assign(left, right):
  if left.shape!=right.shape:
    raise ValueError(f'Shape mismatch. Left Shape: {left.shape}, Right Shape: {right.shape}')
  return torch.nn.Parameter(torch.tensor(right))

In [ ]:
load_weights_into_gpt(gpt, params)
gpt.to(device)

In [ ]:
torch.manual_seed(123)

token_ids=generate(
  model=gpt,
  idx=text_to_token_ids("I want", tokenizer).to(device),
  max_new_tokens=25,
  context_size=NEW_CONFIG["context_length"],
  temp=1.4,
  top_k=50
)

print(f'Output text:\n{token_ids_to_text(token_ids, tokenizer)}')